# CNN_VIT_BILSTM_CROSS_ATTENTION_BASED_TRAFFIC_MANAGEMENT_SYSTEM
## S16 - distil ITD-x into our YOLOv8s (ADR-018)

**What this tests.** ITD v1.2 is a YOLOv8x-class detector (56.8 M parameters,
`imgsz 992`) trained on Indian traffic by IIT Roorkee. On our elevated Dhaka
footage it finds **11-27% more vehicles** than our detector at every confidence
threshold - and runs at **0.8 fps against our 12.5**, missing the latency
requirement by more than an order of magnitude.

Weight transfer is impossible: **0.0%** of our parameters have a shape-compatible
counterpart in theirs. So this uses ITD as a **teacher** instead. It labelled
2,400 frames of real deployment footage offline, where 1.2 s/frame costs
nothing, and the student trains on those labels while keeping its own
architecture and therefore its own speed.

**Two filters that are part of the method, not tidying.**

* ITD has no `e_rickshaw` and no `cattle`. Labels are **merged**: teacher for its
  six shared classes, our detector for those two only.
* ITD hallucinates large vehicles on this view. Rejecting boxes over 5% of frame
  area removed **65% of its bus detections and 43% of its trucks**.

**The acceptance criteria are fixed in ADR-018 and evaluated at the bottom of
this notebook, on real labelled data only.** Pseudo-labels never enter val or
test - the student is trained to agree with the teacher, so agreement is
guaranteed and meaningless.

In [ ]:
import os, sys, random
SEED = 42
random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np, torch
np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("NO GPU. Settings -> Accelerator -> GPU T4 x2.")
print("gpu  ", torch.cuda.get_device_name(0))

# Kaggle's P100 is sm_60 and current PyTorch starts at sm_70 - the card is
# UNUSABLE, not merely slow, and `kernels push` resets the accelerator to P100.
major, minor = torch.cuda.get_device_capability(0)
supported = torch.cuda.get_arch_list()
print("arch  ", "sm_%d%d" % (major, minor), "| build supports", supported)
if ("sm_%d%d" % (major, minor)) not in supported:
    raise SystemExit(
        "INCOMPATIBLE GPU sm_%d%d; build supports %s. "
        "Settings -> Accelerator -> GPU T4 x2 (sm_75)." % (major, minor, supported)
    )

## The code comes from the repository, not from this notebook

In [ ]:
!git clone --depth 1 https://github.com/Divyansh-9/CNN_VIT_BILSTM_CROSS_ATTENTION_BASED_TRAFFIC_MANAGEMENT_SYSTEM.git /kaggle/working/repo 2>&1 | tail -2
%cd /kaggle/working/repo
!pip install -q ultralytics huggingface_hub

## BMD-45 - the elevated training data

Not optional. S11 on IDD alone scored **0.3223** on elevated CCTV; the joint
model scores **0.8941**. ADR-018 criterion 1 is measured on elevated BMD-45, so
training without this source would fail the criterion for a reason that has
nothing to do with the pseudo-labels.

In [ ]:
COUNT = 8000     # matched to the IDD subsample, as S14
import subprocess
subprocess.run([
    "python", "scripts/prepare_bmd45.py",
    "--count", str(COUNT), "--workers", "16",
    "--max-failure-rate", "0.02",
    "--out", "/kaggle/working/bmd45_yolo",
], check=True)

## Locate the attached datasets

Two are needed: the IDD YOLO bootstrap, and the ITD pseudo-labels (which also
carries the S15 checkpoint, because a shallow clone does not pull Git LFS).

In [ ]:
from pathlib import Path
INPUT = Path("/kaggle/input")
candidates = sorted(INPUT.iterdir()) if INPUT.exists() else []
print("mounted:", [c.name for c in candidates] or "NOTHING")
if not candidates:
    raise SystemExit(
        "Nothing under /kaggle/input. Attach BOTH datasets via the Input panel - "
        "a UI Save & Run All uses the draft attachments, not kernel-metadata."
    )

def find_yolo_root(base):
    for path in [base, *base.rglob("*")]:
        if path.is_dir() and (path / "images" / "train").is_dir():
            return path
    return None

IDD = None
for c in candidates:
    IDD = find_yolo_root(c)
    if IDD is not None:
        break
if IDD is None:
    raise SystemExit("no images/train under any attached dataset")
print("IDD    ", IDD)

PSEUDO = None
for c in candidates:
    for labels in c.rglob("labels"):
        if (labels.parent / "images").is_dir() and "train" not in labels.parts:
            PSEUDO = labels.parent
            break
    if PSEUDO is not None:
        break
if PSEUDO is None:
    raise SystemExit("pseudo-label dataset not attached")
print("PSEUDO ", PSEUDO, len(list((PSEUDO / "images").glob("*.jpg"))), "frames")

CKPT = None
for c in candidates:
    for found in c.rglob("s15_*_best.pt"):
        CKPT = found
        break
    if CKPT is not None:
        break
if CKPT is None:
    raise SystemExit("S15 checkpoint not found in the attached datasets")
print("CKPT   ", CKPT)

## Compose - pseudo-labels are a TRAIN source only

`build_joint_dataset.py` writes the two evaluation configs. They are **not**
regenerated with pseudo-labels in them: a criterion whose measuring instrument
moves with the treatment measures nothing.

In [ ]:
import subprocess
subprocess.run([
    "python", "scripts/build_joint_dataset.py",
    "--bmd45", "/kaggle/working/bmd45_yolo",
    "--idd", str(IDD),
    "--out", "/kaggle/working/joint",
], check=True)

subprocess.run([
    "python", "scripts/build_distill_dataset.py",
    "--pseudo", str(PSEUDO),
    "--idd", str(IDD),
    "--bmd45", "/kaggle/working/bmd45_yolo",
    "--out", "/kaggle/working/distill",
], check=True)

print(open("/kaggle/working/distill/distill.yaml").read())

## Train - from the S15 checkpoint, one variable changed

Everything except the data is held at S15 settings: same architecture, imgsz,
batch, seed, and the A31 geometric augmentation. The pseudo-labelled source is
the only difference between this arm and S15, which is what makes the comparison
readable.

In [ ]:
from ultralytics import YOLO

model = YOLO(str(CKPT))
results = model.train(
    data="/kaggle/working/distill/distill.yaml",
    epochs=40, imgsz=640, batch=16, seed=SEED, patience=15,
    perspective=0.0006, degrees=8.0, shear=4.0,   # identical to S15 (A31)
    project="/kaggle/working/runs", name="s16_distill", exist_ok=True, plots=True,
)

## Evaluate - separately, on real labels only

In [ ]:
WEIGHTS = "/kaggle/working/runs/s16_distill/weights/best.pt"
import subprocess

BAR = "=" * 66
for tag, what in (("bmd45", "elevated CCTV"), ("idd", "dashcam")):
    print(); print(BAR); print("  %s  (%s)" % (tag.upper(), what)); print(BAR)
    subprocess.run([
        "python", "scripts/verify_detector_metrics.py",
        "--weights", WEIGHTS,
        "--data", "/kaggle/working/joint/eval_%s.yaml" % tag,
        "--split", "test", "--device", "0",
        "--out", "/kaggle/working/s16_metrics_%s.csv" % tag,
    ], check=True)

## ADR-018 acceptance gate - four criteria, fixed before this run

Criterion 2b exists because **`e_rickshaw` has zero test boxes in IDD and zero in
BMD-45** - it has never been evaluated in any run of this project. A criterion
stated over its AP would be unfalsifiable, so the check is instead whether the
student still predicts the class at all. That cannot show the predictions are
correct; it shows the class has not been silently absorbed into `auto_rickshaw`,
which is the specific damage the label merge exists to prevent.

In [ ]:
import csv, statistics, time
import numpy as np
from pathlib import Path
from ultralytics import YOLO

BASELINE_BMD45 = 0.8941      # S15, experiments/results/s15_metrics_bmd45.csv
BASELINE_IDD = 0.7104        # S14 mean over evaluated classes
BASELINE_CATTLE = 0.3516     # S14, IDD test, 183 boxes

def read(path):
    rows = [r for r in csv.DictReader(open(path)) if r.get("evaluated") == "True"]
    return rows, statistics.fmean(float(r["mAP50"]) for r in rows if r["mAP50"])

bmd_rows, bmd_map = read("/kaggle/working/s16_metrics_bmd45.csv")
idd_rows, idd_map = read("/kaggle/working/s16_metrics_idd.csv")
cattle = None
for r in idd_rows:
    if r["class"] == "cattle":
        cattle = float(r["mAP50"])

# Criterion 3 - latency on THIS host, stated (ADR-003).
student = YOLO(WEIGHTS)
frame = np.random.randint(0, 255, (1080, 1920, 3), dtype=np.uint8)
student.predict(source=frame, imgsz=640, verbose=False)
t0 = time.perf_counter()
for _ in range(20):
    student.predict(source=frame, imgsz=640, verbose=False)
fps = 20 / (time.perf_counter() - t0)

# Criterion 2b - does the student still predict e_rickshaw at all?
before = YOLO(str(CKPT))
pseudo_images = sorted((PSEUDO / "images").glob("*.jpg"))[:200]

def erick_rate(m):
    n = 0
    for image in pseudo_images:
        r = m.predict(source=str(image), imgsz=640, conf=0.25, verbose=False)[0]
        n += sum(1 for c in r.boxes.cls.tolist() if m.names[int(c)] == "e_rickshaw")
    return n

before_n, after_n = erick_rate(before), erick_rate(student)

BAR = "=" * 66
print(BAR); print("  ADR-018 ACCEPTANCE GATE"); print(BAR)
checks = [
    ("1  BMD-45 elevated mAP50 >= 0.8941",
     "%.4f (was %.4f, %+.4f)" % (bmd_map, BASELINE_BMD45, bmd_map - BASELINE_BMD45),
     bmd_map >= BASELINE_BMD45),
    ("2a cattle AP50 drop <= 0.02",
     ("%.4f (was %.4f, %+.4f)" % (cattle, BASELINE_CATTLE, cattle - BASELINE_CATTLE))
     if cattle is not None else "NOT EVALUATED",
     cattle is not None and cattle >= BASELINE_CATTLE - 0.02),
    ("2b e_rickshaw predictions fall <= 50%",
     "%d vs %d on 200 frames" % (after_n, before_n),
     before_n == 0 or after_n >= 0.5 * before_n),
    ("3  >= 10 fps on this host",
     "%.1f fps on %s" % (fps, torch.cuda.get_device_name(0)),
     fps >= 10.0),
    ("4  IDD mAP50 not down > 0.02",
     "%.4f (was %.4f, %+.4f)" % (idd_map, BASELINE_IDD, idd_map - BASELINE_IDD),
     idd_map >= BASELINE_IDD - 0.02),
]
for label, value, ok in checks:
    print("  [%s]  %s" % ("PASS" if ok else "FAIL", label))
    print("          %s" % value)

passed = all(ok for _, _, ok in checks)
print(BAR)
if passed:
    print("  ALL CRITERIA MET - the arm is adoptable.")
else:
    print("  NOT ADOPTED. Record the numbers and the reason; do not retune")
    print("  the criteria to fit the result (ADR-012 discipline).")
print(BAR)
print()
print("  Attribution: distilled from ITD v1.2 (IIT Roorkee), CC BY-NC 4.0.")
print("  This model is a derivative work and stays non-commercial.")